# Stage C in Colab (3 steps)

1. **Runtime → Change runtime type → T4 GPU**
2. **Run Cell 1** – install deps
3. **Run Cell 2** – upload your ZIP (project + pipeline), then upload `stage_b_text_chunks.json` when asked
4. **Run Cell 3** – run Stage C

In [ ]:
# Cell 1: Install (transformers/torch for NER; pymupdf/pdfplumber for pipeline imports)
!pip install -q transformers torch pymupdf pdfplumber

In [ ]:
# Cell 2: Upload project ZIP, then upload stage_b_text_chunks.json
import sys
from google.colab import files
import zipfile
from pathlib import Path

print("Upload your Thesis_llama_colab.zip (or project zip)...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall("/content")
print("ZIP extracted.")

ROOT = Path("/content/Thesis_llama") if (Path("/content") / "Thesis_llama").exists() else Path("/content")
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

print("\nNow upload stage_b_text_chunks.json...")
up2 = files.upload()
stage_b_name = [k for k in up2 if "stage_b" in k and k.endswith(".json")][0]
print(f"Stage B file ready: {stage_b_name}")

In [ ]:
# Cell 3: Run Stage C
import sys
import json
from pathlib import Path
from google.colab import files

ROOT = Path("/content/Thesis_llama") if Path("/content/Thesis_llama").exists() else Path("/content")
sys.path.insert(0, str(ROOT))

STAGE_B_PATH = next(Path("/content").glob("*stage_b*.json"), None) or Path("/content/stage_b_text_chunks.json")
if not STAGE_B_PATH.exists():
    raise FileNotFoundError("No stage_b JSON. Run Cell 2 and upload stage_b_text_chunks.json.")

from pipeline.data import TextChunks, StatementsWithMedicalEntities
from pipeline.models import NeuralModel
from pipeline.devices import RecognizeEntities

with open(STAGE_B_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
chunks = TextChunks()
for c in data["chunks"]:
    chunks.add_chunk(page=c["page"], text=c["text"], source=c.get("source", ""), chunk_id=c.get("chunk_id"))

model = NeuralModel(model_name="d4data/biomedical-ner-all")
recognizer = RecognizeEntities(neural_model=model, min_score=0.55, acronym_file=None)
result = recognizer.infer(chunks)

out_path = Path("/content/stage_c_statements_with_entities.json")
out_path.write_text(json.dumps({"metadata": {"stage": "c", "total_statements": result.count(), "total_entities": result.get_entity_count()}, "statements": result.get_all()}, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Done. Statements: {result.count()}, Entities: {result.get_entity_count()}")
print(f"Output: {out_path}")
files.download(str(out_path))